In [5]:
from paths import path
import os
import glob

In [ ]:
# From Tables 1-3 of the paper.
mass_to_stellar_B = {
    0.0899: 1000,
    0.1: 987.0,
    0.2: 858.2,
    0.3: 729.5,
    0.4: 600.8,
    0.5: 472.1,
    0.6: 343.4,
    0.7: 214.7,
    0.8: 85.99,
    0.86: 8.491,
    0.9: 7.686,
    1.0: 5.000,
    1.1: 2.314,
    1.2: 2.314
}

mass_to_XUV_frac = {
    0.0899: 3.02993108e-3,
    0.1: 2.827e-3,
    0.2: 2.624e-3,
    0.3: 2.421e-3,
    0.4: 2.218e-3,
    0.5: 2.015e-3,
    0.6: 1.812e-3,
    0.7: 1.609e-3,
    0.8: 1.406e-3,
    0.86: 1.203e-3,
    0.9: 1.203e-3,
    1.0: 1e-3,
    1.1: 0.8012e-4,
    1.2: 0.5012e-4
}

mass_to_beta_XUV = {
    0.0899: 1.17241206,
    0.1: 1.176,
    0.2: 1.182,
    0.3: 1.188,
    0.4: 1.194,
    0.5: 1.20,
    0.6: 1.206,
    0.7: 1.212,
    0.8: 1.218,
    0.86: 1.221,
    0.9: 1.224,
    1.0: 1.23,
    1.1: 1.24,
    1.2: 1.25
}

mass_to_sat_time_yr = {
    0.0899: -3.14617755,
    0.1: 2e9,
    0.2: 1e9,
    0.3: 5e8,
    0.4: 3e8,
    0.5: 1e8,
    0.6: 1e8,
    0.7: 1e8,
    0.8: 1e8,
    0.86: 1e8,
    0.9: 1e8,
    1.0: 1e8,
    1.1: 1e7,
    1.2: 1e7
}

label_to_stellar_mass = {
    'trappist': 0.0899,
    'm0p1': 0.1,
    'm0p2': 0.2,
    'm0p3': 0.3,
    'm0p4': 0.4,
    'm0p5': 0.5,
    'm0p6': 0.6,
    'm0p7': 0.7,
    'm0p8': 0.8,
    'kv': 0.86,
    'm0p9': 0.9,
    'sun': 1,
    'm1p1': 1.1,
    'm1p2': 1.2,
    'kv_test': 0.1
}

In [ ]:
synthesis_simulation_directories = sorted(glob.glob(path("src", "parameter_sweeps", "synthesis", "*")))

stellar_labels = label_to_stellar_mass.keys()

for sim in synthesis_simulation_directories:
    stellar_label = sim.split(os.sep)[-1].replace("_atm", "").replace("_water", "")

    if stellar_label in stellar_labels:
        stellar_mass = label_to_stellar_mass[stellar_label]
        stellar_B = mass_to_stellar_B[stellar_mass]

        input_files = glob.glob(path(sim, "*.in"))

        for input_file in input_files:
            if ("vpl" not in input_file and "earth" not in input_file):
                with open(input_file, "r") as file:                    
                    contents = file.read()

                    mass = round(float(contents.split("dMass")[1].split("\n")[0].replace(" ", "").split("#")[0])/2e30, 2)
                    XUV_beta = float(contents.split("dXUVBeta")[1].split("\n")[0].replace(" ", "").split("#")[0])
                    XUV_frac = float(contents.split("dSatXUVFrac")[1].split("\n")[0].replace(" ", "").split("#")[0])
                    sat_time_yr = float(contents.split("dSatXUVTime")[1].split("\n")[0].replace(" ", "").split("#")[0])
                    stellar_B_from_file = round(float(contents.split("dSurfMagField")[1].split("\n")[0].replace(" ", "").split("#")[0]) * 1e4, 2)

                    if round(stellar_mass - mass, 2) != 0:
                        print(sim, "Mass does not match Table data. Mass is", mass, "but should be", stellar_mass)
                    
                    if round(XUV_frac - mass_to_XUV_frac[stellar_mass], 2) != 0:
                        print(sim, "XUV fraction does not match Table data. XUV frac = ", XUV_frac, "while it should be", mass_to_XUV_frac[stellar_mass])

                    if round(XUV_beta - mass_to_beta_XUV[stellar_mass], 2) != 0:
                        print(sim, "XUV beta does not match Table data.")

                    if round(sat_time_yr - mass_to_sat_time_yr[stellar_mass], 2) != 0:
                        print(sim, "Saturation time does not match Table data")

                    if round(stellar_B - stellar_B_from_file, 2) != 0:
                        print(sim, "Stellar magnetic field does not match Table data. B =", stellar_B_from_file, "but should be", stellar_B)